In [2]:
# Imports
import sys
import os
sys.path.append(os.path.abspath('../..'))


from controller.marl.main import setup
from controller.marl.core.config import Config
from controller.marl.runners.sim_runner import run_sim


from project_paths import PROJECT_ROOT


import torch
from controller.marl.core.datasets import FilteredObsData
from torch.utils.data import DataLoader
import torch.nn.functional as F


from plt_style import set_style
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

set_style()

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
config = Config.from_yaml(PROJECT_ROOT / "configs")
assert config.training.seed != -1
assert config.comms.communication_type.value == "AIM", f"Comm type is {config.comms.communication_type}, should be aim"
assert config.comms.autoencoder_type != "hq-vae"

In [8]:
system, config = setup(config, device, load_agent_architecture=True)

Optimizer parameter groups changed. Starting with fresh optimizer states.
Loaded checkpoint from step 52000


In [9]:
actor = system["actor"]
actor.eval()

PPO_Actor(
  (comm_protocol): AimComms(
    (encoder): Encoder(
      (encoder): Sequential(
        (0): Linear(in_features=36, out_features=512, bias=True)
        (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (2): ReLU()
        (3): Linear(in_features=512, out_features=256, bias=True)
        (4): ReLU()
        (5): Linear(in_features=256, out_features=8, bias=True)
      )
      (latent_handler): VQ_VAE(
        (embeddings): ModuleList(
          (0): Embedding(32, 8)
        )
      )
    )
    (predict_agents_returns): Linear(in_features=520, out_features=2, bias=True)
    (predict_critic_value): Linear(in_features=520, out_features=1, bias=True)
    (comm_heads): ModuleList(
      (0): Linear(in_features=512, out_features=32, bias=True)
    )
    (predict_intent_sender_goal): Linear(in_features=520, out_features=2, bias=True)
    (predict_intent_receiver_goal): Linear(in_features=520, out_features=2, bias=True)
  )
  (body): Sequential(
    (0): Linear(in

In [10]:
obs_logs_file = "./temp.csv"

In [11]:
run_sim(system, config, device, 2, collect_obs_file=obs_logs_file)

Running Simulation:   0%|          | 0/2 [00:00<?, ?it/s]


TypeError: Simulation.parallel_step() missing 1 required positional argument: 'flat_world_comms'

In [ ]:
GO = system["sim"].get_global_obs_dim() + system["sim"].get_num_agents() * config.comms.communication_size * config.comms.num_comms
mask = torch.tensor(system["sim"].get_agent_external_obs_mask(0), dtype=torch.bool, device=device)
dataset = FilteredObsData(obs_logs_file, system["act_shape"][0], GO, mask, device)

dataloader = DataLoader(dataset, batch_size=config.aim_training.aim_batch_size)

In [ ]:
co_occurrence_matrix = np.zeros((config.comms.vocab_size, 5))

for batch_obs, batch_global_obs, batch_actions, batch_targets, batch_critic_values, batch_return_values in dataloader:
    
    batch_obs = batch_obs.to(device)
    batch_actions = batch_actions.to(device).long()
    
    actor_hidden_states = actor.init_hidden(batch_size=batch_obs.shape[0])
    
    action_logits, lstm_output, _ = actor(batch_obs, actor_hidden_states, None)
    
    _, to_save, _ = actor.comm_protocol.get_comms_during_rollout(lstm_output)
    
    code_indicies = to_save[..., 0].flatten().cpu().numpy().astype(int)
    actions = batch_actions.flatten().cpu().numpy().astype(int)
    
    for code, action in zip(code_indicies, actions):
        co_occurrence_matrix[code, action] += 1

In [ ]:
code_usage_sums = co_occurrence_matrix.sum(axis=1)
active_code_indices = np.where(code_usage_sums > 100)[0] 

filtered_matrix = co_occurrence_matrix[active_code_indices]

row_sums = filtered_matrix.sum(axis=1, keepdims=True)
normalized_matrix = np.divide(filtered_matrix, row_sums, out=np.zeros_like(filtered_matrix), where=row_sums != 0)

actions = {
    0: "Step North",
    1: "Step South",
    2: "Step West",
    3: "Step East",
    4: "Interact"
}
action_labels = [actions[i] for i in range(5)] 
code_labels = [f"Code {i}" for i in active_code_indices]

df_cm = pd.DataFrame(normalized_matrix, index=code_labels, columns=action_labels)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_cm, annot=True, cmap="Blues", fmt=".2f", cbar_kws={'label': 'Probability: P(Action | Code)'})

plt.title("Emergent Semantics: Broadcasted AIM Code vs. Physical Action", fontsize=14, pad=15)
plt.xlabel("Physical Agent Action", fontsize=12)
plt.ylabel("Broadcasted AIM Code", fontsize=12)
plt.yticks(rotation=0) 

plt.tight_layout()
plt.show()

In [ ]:
possible_targets = config.simulation.arena_width * config.simulation.arena_width
co_occurrence_matrix = np.zeros((config.comms.vocab_size, possible_targets))

for batch_obs, batch_global_obs, batch_actions, batch_targets, batch_critic_values, batch_return_values in dataloader:
    
    batch_obs = batch_obs.to(device)
    batch_targets = batch_targets.to(device).long()
    
    actor_hidden_states = actor.init_hidden(batch_size=batch_obs.shape[0])
    
    action_logits, lstm_output, _ = actor(batch_obs, actor_hidden_states, None)
    
    _, to_save, _ = actor.comm_protocol.get_comms_during_rollout(lstm_output)
    
    codes = to_save[..., 0].flatten().cpu().numpy().astype(int)
    
    norm_x = batch_targets[..., 0].flatten().cpu().numpy()
    norm_y = batch_targets[..., 1].flatten().cpu().numpy()
    
    disc_x = np.round(norm_x * config.simulation.arena_width).astype(int)
    disc_y = np.round(norm_y * config.simulation.arena_width).astype(int)
    
    target_ids = (disc_y * config.simulation.arena_width) + disc_x

    for code, target in zip(codes, target_ids):
        co_occurrence_matrix[code, target] += 1

In [ ]:
code_usage_sums = co_occurrence_matrix.sum(axis=1)
active_code_indices = np.where(code_usage_sums > 100)[0] 

filtered_matrix = co_occurrence_matrix[active_code_indices]

row_sums = filtered_matrix.sum(axis=1, keepdims=True)
normalized_matrix = np.divide(filtered_matrix, row_sums, out=np.zeros_like(filtered_matrix), where=row_sums != 0)


target_labels = [f"Target ({i % config.simulation.arena_width}, {i // config.simulation.arena_width})" for i in range(possible_targets)] 
code_labels = [f"Code {i}" for i in active_code_indices]

df_cm = pd.DataFrame(normalized_matrix, index=code_labels, columns=action_labels)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_cm, annot=True, cmap="Blues", fmt=".2f", cbar_kws={'label': 'Probability: P(Action | Code)'})

plt.title("Emergent Semantics: Broadcasted AIM Code vs. Physical Action", fontsize=14, pad=15)
plt.xlabel("Physical Agent Action", fontsize=12)
plt.ylabel("Broadcasted AIM Code", fontsize=12)
plt.yticks(rotation=0) 

plt.tight_layout()
plt.show()